# Chart Classifier — Training su Google Colab

Esegui le celle **in ordine**. Alla prima esecuzione clona il repo e scarica i dati.

Dopo una **disconnessione**, ri-esegui dalla cella **3 (Configurazione)** in poi:
il resume dal checkpoint su Drive è automatico — non si riparte da zero.

In [ ]:
# 1. VERIFICA AMBIENTE
import torch

print(f'PyTorch:  {torch.__version__}')
print(f'CUDA:     {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:      {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM:     {vram:.1f} GB')
else:
    print('ATTENZIONE: CUDA non disponibile. Vai su Runtime > Cambia tipo di runtime > GPU.')

In [ ]:
# 2. DIPENDENZE
!pip install timm tqdm --quiet
print('Dipendenze OK.')

In [ ]:
# 3. CONFIGURAZIONE — modifica solo questa cella
# ============================================================
# Nome del run: identifica questa sessione di training su Drive
RUN_NAME = 'chart_classifier_fra'

# Google Drive — struttura attesa:
#   GDRIVE_BASE/
#   ├── chart_classifier.zip    <- codice + dati (src/ e data/ alla radice dello zip)
#   └── runs/
#       └── {RUN_NAME}/
#           ├── best_model.pth
#           ├── last_checkpoint.pth
#           └── confusion_matrix_*.png
GDRIVE_BASE        = '/content/drive/MyDrive/chart_classifier'
GDRIVE_PROJECT_ZIP = f'{GDRIVE_BASE}/chart_classifier.zip'  # codice + dati
GDRIVE_RUNS        = f'{GDRIVE_BASE}/runs'

LOCAL_PROJECT = '/content/chart_classifier'

# Iperparametri training
EPOCHS      = 30
BATCH_SIZE  = 64
LR          = 5e-5
WD          = 0.05
NUM_WORKERS = 4

print(f'Run:   {RUN_NAME}')
print(f'Drive: {GDRIVE_RUNS}/{RUN_NAME}')

In [ ]:
# 4. MOUNT GOOGLE DRIVE
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 5. ESTRAI PROGETTO DA DRIVE (codice + dati)
# Lo zip deve avere src/ e data/ alla radice (senza cartella wrapper).
# Se aggiorni il codice su Drive, elimina LOCAL_PROJECT e riesegui questa cella.
import os

if not os.path.exists(f'{LOCAL_PROJECT}/src/train.py'):
    print('Estrazione da Drive...')
    os.makedirs(LOCAL_PROJECT, exist_ok=True)
    !unzip -q {GDRIVE_PROJECT_ZIP} -d {LOCAL_PROJECT}
    print('Estrazione completata.')
else:
    n_train = sum(len(fs) for _, _, fs in os.walk(f'{LOCAL_PROJECT}/data/train'))
    n_val   = sum(len(fs) for _, _, fs in os.walk(f'{LOCAL_PROJECT}/data/val'))
    print(f'Progetto già presente — train: {n_train} img, val: {n_val} img')

%cd {LOCAL_PROJECT}
print(f'Working dir: {os.getcwd()}')

In [ ]:
# 6. TRAINING
# best_model.pth e last_checkpoint.pth sono salvati direttamente su Drive.
# Dopo una disconnessione, ri-esegui le celle 3-6 e poi questa: il resume è automatico.
import os

os.makedirs(f'{GDRIVE_RUNS}/{RUN_NAME}', exist_ok=True)

checkpoint_path = f'{GDRIVE_RUNS}/{RUN_NAME}/last_checkpoint.pth'
resume_flag = f'--resume {checkpoint_path}' if os.path.isfile(checkpoint_path) else ''

if resume_flag:
    print(f'[RESUME] Checkpoint trovato su Drive: {checkpoint_path}')
else:
    print('[NUOVO]  Nessun checkpoint su Drive — training da zero.')

!python src/train.py \
    --name {RUN_NAME} \
    --data_dir ./data \
    --save_dir {GDRIVE_RUNS} \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --lr {LR} \
    --wd {WD} \
    --num_workers {NUM_WORKERS} \
    --device cuda \
    {resume_flag}

---
## Inferenza su directory di immagini

Esegui questa sezione in modo indipendente dal training.
Assicurati di aver eseguito le celle 3 (Config), 4 (Drive) e 5 (Clone).

In [ ]:
# 7. CONFIGURAZIONE INFERENZA
INFERENCE_MODEL   = f'{GDRIVE_RUNS}/{RUN_NAME}/best_model.pth'
INFERENCE_TARGET  = f'{GDRIVE_BASE}/images_to_classify'  # cartella o immagine singola
INFERENCE_BATCH   = 128
INFERENCE_WORKERS = 4

In [ ]:
# 8. ESEGUI INFERENZA
!python src/inference.py \
    --target {INFERENCE_TARGET} \
    --model_path {INFERENCE_MODEL} \
    --batch_size {INFERENCE_BATCH} \
    --workers {INFERENCE_WORKERS}